In [105]:
import pandas as pd
import numpy as np
import pyarrow as pa

# para viz
import altair as alt

# desactivar vegafusion y el renderer HTML de Altair para este entorno
alt.data_transformers.enable('default')

# para el calculo del IVF
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# para el calculo del PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [106]:
df = pd.read_csv(
    'indicadores_mensuales.csv',
    encoding='latin1',
    sep=';',
    decimal=','
)

# Traducir nombres de columnas al inglés
translation_dict = {
    'anio': 'year',
    'periodo': 'period',
    'mes': 'month',
    'area': 'area',
    'ciudad': 'city',
    'tasa_participacion_global': 'global_participation_rate',
    'tasa_participacion_bruta': 'gross_participation_rate',
    'tasa_desempleo': 'unemployment_rate',
    'empleo_total': 'total_employment',
    'empleo_formal': 'formal_employment',
    'empleo_informal': 'informal_employment',
    'empleo_adecuado': 'adequate_employment',
    'subempleo': 'underemployment',
    'no_remunerado': 'unpaid_work',
    'otro_no_pleno': 'other_non_full_employment',
    'brecha_adecuado_HM': 'adequate_employment_gap_hm',
    'brecha_salarial_HM': 'salary_gap_hm',
    'NiNi': 'neet',
    'desempleo_juvenil': 'youth_unemployment',
    'trabajo_infantil': 'child_labor',
    'empleo_manufactura': 'manufacturing_employment',
    'tasa_asistencia_clases': 'school_attendance_rate',
    'pobreza_ingresos': 'income_poverty',
    'pobreza_extrema_ingresos': 'extreme_income_poverty'
}

# Renombrar columnas que existan
for old_name, new_name in translation_dict.items():
    if old_name in df.columns:
        df = df.rename(columns={old_name: new_name})

df.columns = df.columns.str.lower()

# Tratar ciudad como cadena de caracteres
if 'city' in df.columns:
    df['city'] = df['city'].astype('Int64').astype(str).str.zfill(6)

id_str_cols = ['city', 'month']
for col in df.columns:
    if col not in id_str_cols:
        df[col] = df[col].astype('Float64')
df.head()

,year,period,month,area,city,global_participation_rate,gross_participation_rate,unemployment_rate,total_employment,formal_employment,...,other_non_full_employment,adequate_employment_gap_hm,salary_gap_hm,neet,youth_unemployment,child_labor,manufacturing_employment,school_attendance_rate,income_poverty,extreme_income_poverty
0,2007.0,6.0,Junio,1.0,010150,69.6709,50.6597,5.6692,94.3308,67.7679,...,14.9143,32.5062,38.3322,8.8601,8.4198,5.0324,19.5232,79.0481,12.9154,3.4991
1,2007.0,6.0,Junio,1.0,070150,69.5793,47.9878,6.39,93.61,44.5163,...,18.088,24.5242,19.8952,18.1669,8.9962,4.7061,8.7018,73.8733,23.7168,7.3746
2,2007.0,6.0,Junio,1.0,090150,70.2874,50.1103,8.9783,91.0217,53.3911,...,15.6068,32.5023,28.153,17.665,15.9064,6.1991,13.7591,75.6331,19.1637,6.132
3,2007.0,6.0,Junio,1.0,170150,68.7131,50.6626,5.9597,94.0403,68.125,...,14.0037,32.6926,47.4736,11.6609,9.6217,4.873,14.9477,77.9396,11.7764,3.4935
4,2007.0,6.0,Junio,1.0,180150,67.9143,50.2609,4.2956,95.7044,68.6879,...,16.9191,33.5418,32.4586,9.4014,8.6598,5.4683,18.4217,81.4531,14.2329,4.2952


In [107]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        835 non-null    Float64
 1   period                      835 non-null    Float64
 2   month                       835 non-null    str    
 3   area                        835 non-null    Float64
 4   city                        835 non-null    str    
 5   global_participation_rate   835 non-null    Float64
 6   gross_participation_rate    835 non-null    Float64
 7   unemployment_rate           835 non-null    Float64
 8   total_employment            835 non-null    Float64
 9   formal_employment           835 non-null    Float64
 10  informal_employment         835 non-null    Float64
 11  adequate_employment         835 non-null    Float64
 12  underemployment             835 non-null    Float64
 13  unpaid_work                 835 non-null    Fl

In [108]:
df.describe()

,year,period,area,global_participation_rate,gross_participation_rate,unemployment_rate,total_employment,formal_employment,informal_employment,adequate_employment,...,other_non_full_employment,adequate_employment_gap_hm,salary_gap_hm,neet,youth_unemployment,child_labor,manufacturing_employment,school_attendance_rate,income_poverty,extreme_income_poverty
count,835.0,835.0,835.0,835.0,835.0,835.0,835.0,835.0,835.0,835.0,...,835.0,828.0,831.0,828.0,824.0,827.0,835.0,834.0,835.0,835.0
mean,2018.675449,7.037126,1.299401,66.421,49.749414,5.140647,94.859353,60.185546,32.689166,48.424601,...,23.395945,29.677616,22.437376,16.677933,10.678949,2.053821,13.906131,77.996113,12.788551,3.596361
std,5.46419,3.531946,0.45827,7.342329,6.954126,3.774066,3.774066,15.766562,14.825057,13.165283,...,9.736476,43.426929,23.886686,12.092971,9.726943,6.823578,7.326927,9.443054,10.594457,5.010565
min,2007.0,1.0,1.0,38.4615,21.2766,0.0,69.2308,0.0,0.0,0.0,...,0.0,-641.1434,-197.0297,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,2014.0,4.0,1.0,62.5,46.3561,3.1732,93.1122,51.8337,22.33785,42.14425,...,17.8999,20.995425,15.48975,10.4446,6.0178,0.0,9.65825,75.193925,7.355,0.8039
50%,2021.0,6.0,1.0,65.2478,48.8146,4.6611,95.3389,63.3186,28.9503,49.4147,...,21.4286,27.9088,22.6154,15.7354,9.66995,0.4858,14.2857,79.5962,10.9077,2.4604
75%,2023.0,10.0,2.0,69.3602,51.86475,6.8878,96.8268,71.7984,39.96175,57.09925,...,25.8383,36.469375,30.18175,20.361575,13.8404,1.85515,18.39135,82.60005,16.4557,4.06515
max,2026.0,12.0,2.0,100.0,86.6667,30.7692,100.0,100.0,100.0,100.0,...,76.9231,100.0,100.0,100.0,100.0,100.0,39.5074,100.0,91.0714,50.0


In [ ]:
id_cols = ['year','period','month','area', 'city']
num_cols = [c for c in df.columns if c not in id_cols]
df[num_cols].head()

,year,period,month,area,city,global_participation_rate,gross_participation_rate,unemployment_rate,total_employment,formal_employment,...,other_non_full_employment,adequate_employment_gap_hm,salary_gap_hm,neet,youth_unemployment,child_labor,manufacturing_employment,school_attendance_rate,income_poverty,extreme_income_poverty
0,2007.0,6.0,Junio,1.0,010150,69.6709,50.6597,5.6692,94.3308,67.7679,...,14.9143,32.5062,38.3322,8.8601,8.4198,5.0324,19.5232,79.0481,12.9154,3.4991
1,2007.0,6.0,Junio,1.0,070150,69.5793,47.9878,6.39,93.61,44.5163,...,18.088,24.5242,19.8952,18.1669,8.9962,4.7061,8.7018,73.8733,23.7168,7.3746
2,2007.0,6.0,Junio,1.0,090150,70.2874,50.1103,8.9783,91.0217,53.3911,...,15.6068,32.5023,28.153,17.665,15.9064,6.1991,13.7591,75.6331,19.1637,6.132
3,2007.0,6.0,Junio,1.0,170150,68.7131,50.6626,5.9597,94.0403,68.125,...,14.0037,32.6926,47.4736,11.6609,9.6217,4.873,14.9477,77.9396,11.7764,3.4935
4,2007.0,6.0,Junio,1.0,180150,67.9143,50.2609,4.2956,95.7044,68.6879,...,16.9191,33.5418,32.4586,9.4014,8.6598,5.4683,18.4217,81.4531,14.2329,4.2952


In [110]:
df[num_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   global_participation_rate   835 non-null    Float64
 1   gross_participation_rate    835 non-null    Float64
 2   unemployment_rate           835 non-null    Float64
 3   total_employment            835 non-null    Float64
 4   formal_employment           835 non-null    Float64
 5   informal_employment         835 non-null    Float64
 6   adequate_employment         835 non-null    Float64
 7   underemployment             835 non-null    Float64
 8   unpaid_work                 835 non-null    Float64
 9   other_non_full_employment   835 non-null    Float64
 10  adequate_employment_gap_hm  828 non-null    Float64
 11  salary_gap_hm               831 non-null    Float64
 12  neet                        828 non-null    Float64
 13  youth_unemployment          824 non-null    Fl

# EDA

In [123]:
#df_long = df[num_cols].melt(var_name='variable', value_name='value')
df_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 15865 entries, 0 to 15864
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   variable  15865 non-null  str    
 1   value     15827 non-null  Float64
dtypes: Float64(1), str(1)
memory usage: 263.5 KB


In [ ]:
# Usar una muestra muy pequeña para evitar problemas de renderizado
small_df = df_filtered.sample(n=min(len(df_filtered), 15865), random_state=42)

chart_hist = alt.Chart(small_df).mark_bar().encode(
    x=alt.X(
        'value:Q',
        bin=alt.Bin(maxbins=20),
        scale=alt.Scale(domain=[0, 100]),
        title='Percentage (%)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    y=alt.Y('count():Q', title='Frequency')
).properties(
    width=180,
    height=120
).facet(
    facet=alt.Facet('variable:N', title='Distributions by variable'),
    columns=3
)

chart_hist

MaxRowsError: The number of rows in your dataset (15748) is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

Or, see https://altair-viz.github.io/user_guide/large_datasets.html for additional information
on how to plot large datasets.

alt.FacetChart(...)

In [113]:
small_df_box = df_filtered.sample(n=min(len(df_filtered), 1000), random_state=42)

orden_mediana = small_df_box.groupby('variable')['value'].median().sort_values().index.tolist()

chart_box = alt.Chart(small_df_box).mark_boxplot().encode(
    x=alt.X(
        'value:Q',
        scale=alt.Scale(domain=[0, 100]),
        title='Percentage (%)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    y=alt.Y(
        'variable:N',
        sort=orden_mediana,
        title=None
    )
).properties(
    width=700,
    height=500,
    title='Boxplots por variable'
)

chart_box

alt.Chart(...)

# Correlación de Pearson

In [12]:
corr = df[num_cols].corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

corr_pairs = (
    corr.where(mask)
        .stack()
        .reset_index()
)

corr_pairs.columns = ['variable_1', 'variable_2', 'pearson_r']
corr_pairs['abs_r'] = corr_pairs['pearson_r'].abs()

corr_pairs = corr_pairs.sort_values('abs_r', ascending=False).reset_index(drop=True)

corr_pairs.head(30)

,variable_1,variable_2,pearson_r,abs_r
0,tasa_desempleo,empleo_total,-1.000000,1.000000
1,empleo_formal,empleo_informal,-0.979382,0.979382
2,empleo_total,desempleo_juvenil,-0.951044,0.951044
3,tasa_desempleo,desempleo_juvenil,0.951044,0.951044
4,pobreza_multidimensional,necesidades_basicas_insatisfechas,0.948968,0.948968
5,empleo_adecuado,subempleo,-0.906090,0.906090
6,pobreza_multidimensional,pobreza_extrema_multidimensional,0.898964,0.898964
7,pobreza_ingresos,pobreza_extrema_ingresos,0.881455,0.881455
8,empleo_formal,tasa_asistencia_clases,0.859146,0.859146
9,pobreza_extrema_multidimensional,necesidades_basicas_insatisfechas,0.845212,0.845212


In [13]:
corr = df[num_cols].corr(method='pearson')

corr_long = (
    corr.reset_index()
    .melt(id_vars='index', var_name='variable_2', value_name='correlacion')
    .rename(columns={'index': 'variable_1'})
)

orden_vars = corr.columns.tolist()
idx_map = {var: i for i, var in enumerate(orden_vars)}

corr_long['i'] = corr_long['variable_1'].map(idx_map)
corr_long['j'] = corr_long['variable_2'].map(idx_map)

# solo triángulo inferior
corr_long_inf = corr_long[corr_long['i'] < corr_long['j']]

base = alt.Chart(corr_long_inf).encode(
    x=alt.X('variable_1:N', sort=orden_vars, title=None),
    y=alt.Y('variable_2:N', sort=orden_vars, title=None),
    tooltip=[
        alt.Tooltip('variable_1:N', title='Variable 1'),
        alt.Tooltip('variable_2:N', title='Variable 2'),
        alt.Tooltip('correlacion:Q', format='.3f', title='Correlación')
    ]
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        'correlacion:Q',
        scale=alt.Scale(
            domain=[-1, 0, 1],
            range=['#b2182b', '#f7f7f7', '#2166ac']
        ),
        title='r de Pearson'
    )
)

text = base.mark_text(fontSize=10).encode(
    text=alt.Text('correlacion:Q', format='.2f'),
    color=alt.condition(
        'abs(datum.correlacion) >= 0.5',
        alt.value('white'),
        alt.value('black')
    )
)

chart = (heatmap + text).properties(
    width=600,
    height=600,
    title='Matriz de correlación de Pearson'
).configure_view(
    strokeWidth=0
)

chart

alt.LayerChart(...)

# Variance Inflation Factor (VIF)

In [14]:
exclude_cols = ['p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia']
X = df.drop(columns=exclude_cols).copy()
X = X.apply(pd.to_numeric, errors='coerce').dropna()

X_const = sm.add_constant(X)

vif_df = pd.DataFrame({
    'variable': X_const.columns,
    'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
})

vif_df['tolerancia'] = 1 / vif_df['VIF']
vif_df

/Users/erickedu85/Dropbox/python_projects/enemdu_icm/.venv/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variable,VIF,tolerancia
0,const,inf,0.000000
1,tasa_participacion_global,16.221656,0.061646
2,tasa_participacion_bruta,25.074239,0.039882
3,tasa_desempleo,inf,0.000000
4,empleo_total,inf,0.000000
5,empleo_formal,79.765386,0.012537
6,empleo_informal,72.236874,0.013843
7,empleo_adecuado,106.539948,0.009386
8,subempleo,65.496673,0.015268
9,no_remunerado,18.769322,0.053278


# PCA

In [15]:
# se elimina empleo_total por alta colinealidad con tasa_desempleo
cols_excluir_pca = [
    'p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia', 'empleo_total'
]

X = df.drop(columns=cols_excluir_pca).copy()
X = X.apply(pd.to_numeric, errors='coerce').dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

## Varianza

In [16]:
explained_variance = pd.DataFrame({
    'componente': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'varianza_explicada': pca.explained_variance_ratio_,
    'varianza_acumulada': pca.explained_variance_ratio_.cumsum()
})

explained_variance

,componente,varianza_explicada,varianza_acumulada
0,PC1,0.385731,0.385731
1,PC2,0.207440,0.593171
2,PC3,0.150986,0.744157
3,PC4,0.098188,0.842345
4,PC5,0.043597,0.885942
5,PC6,0.028923,0.914865
6,PC7,0.023633,0.938499
7,PC8,0.016172,0.954670
8,PC9,0.010794,0.965464
9,PC10,0.008235,0.973699


## Cargas para PC1

In [17]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(X.shape[1])],
    index=X.columns
)

loadings[['PC1']].sort_values('PC1')

,PC1
empleo_informal,-0.316933
pobreza_multidimensional,-0.301161
necesidades_basicas_insatisfechas,-0.297699
nini,-0.288538
pobreza_extrema_multidimensional,-0.257277
pobreza_ingresos,-0.251358
subempleo,-0.217174
brecha_adecuado_hm,-0.175644
pobreza_extrema_ingresos,-0.145579
desempleo_juvenil,-0.138268


## Construir el Indice Compuesto Multivariado (ICM) solo con el PC1

 - 0 = peor ICM relativo observado
 - 100 = mejor ICM relativo observado

In [18]:
indice_compuesto_multivariado = X_pca[:, 0]

indice_df = df.loc[X.index, ['p_anio', 'p_ciudad', 'provincia', 'canton', 'parroquia']].copy()
indice_df['indice_compuesto_multivariado'] = indice_compuesto_multivariado

indice_min = indice_df['indice_compuesto_multivariado'].min()
indice_max = indice_df['indice_compuesto_multivariado'].max()

# Transformación min-max del indice
indice_df['indice_compuesto_multivariado_0_100'] = (
    (indice_df['indice_compuesto_multivariado'] - indice_min) / (indice_max - indice_min)
) * 100

indice_df

,p_anio,p_ciudad,provincia,canton,parroquia,indice_compuesto_multivariado,indice_compuesto_multivariado_0_100
0,2007,010150,AZUAY,CUENCA,CUENCA,2.221896,81.303897
1,2007,070150,EL ORO,MACHALA,MACHALA,-5.623312,3.960425
2,2007,090150,GUAYAS,GUAYAQUIL,GUAYAQUIL,-3.782332,22.110071
3,2007,170150,PICHINCHA,DISTRITO METROPOLITANO DE QUITO,QUITO,2.116305,80.262905
4,2007,180150,TUNGURAHUA,AMBATO,AMBATO,1.519539,74.379575
...,...,...,...,...,...,...,...
90,2025,010150,AZUAY,CUENCA,CUENCA,2.550913,84.547566
91,2025,070150,EL ORO,MACHALA,MACHALA,-2.504504,34.707786
92,2025,090150,GUAYAS,GUAYAQUIL,GUAYAQUIL,-3.029002,29.536922
93,2025,170150,PICHINCHA,DISTRITO METROPOLITANO DE QUITO,QUITO,0.962578,68.888669


## Series temporales del Indice Compuesto Multivariado

In [19]:
indice_df['anio_fecha'] = pd.to_datetime(indice_df['p_anio'].astype(str), format='%Y')

chart = alt.Chart(indice_df).mark_line(point=True).encode(
    x=alt.X(
        'anio_fecha:T',
        title='Año',
        scale=alt.Scale(nice=False),
        axis=alt.Axis(format='%Y')
    ),
    y=alt.Y(
        'indice_compuesto_multivariado_0_100:Q', 
        title='Índice Compuesto Multivariado (0-100)',
        axis=alt.Axis(format='.0f', labelExpr="datum.label + '%'")
    ),
    color=alt.Color(
        'parroquia:N', 
        title='Parroquia'
    ),
    tooltip=[
        alt.Tooltip('p_anio:O', title='Año'),
        alt.Tooltip('provincia:N'),
        alt.Tooltip('canton:N'),
        alt.Tooltip('parroquia:N'),
        alt.Tooltip('indice_compuesto_multivariado_0_100:Q', format='.2f', title='Índice')
    ]
).properties(
    width=700,
    height=400,
    title='Evolución del Índice Compuesto Multivariado (ICM) a lo largo del tiempo'
)

chart

alt.Chart(...)